# Thesis — 150-sentence journal profile (10 training seeds)

1. **Runtime:** Google Colab with **GPU** (recommended).
2. **Repo:** public — no GitHub token.
3. **Profile:** `--journal-paper` (Exp04 λ grid, Exp01/04 + OOF SVM/RF fusion, 5-fold CV).
4. **Seeds:** `--num-seeds 10` (overrides journal default of 3).
5. **Data:** place `ner_dataset.pkl` or `ner_dataset.csv` under `data/` in the cloned repo.
6. **Outputs:** everything writes to **`5folds_run/`** on Drive (separate from older cross-comparison folders).

Docs: `thesis_overview.md` Part IV, `COLAB_README.md`.

## 1 — Mount Google Drive (recommended)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2 — Paths (edit `WORK_ROOT` if needed)

In [ ]:
import os
from pathlib import Path

WORK_ROOT = Path("/content/drive/MyDrive/thesis_project")
REPO_DIR = WORK_ROOT / "thesis"
REPO_URL = "https://github.com/menachemsperka/thesis.git"

WORK_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_ROOT)
print("WORK_ROOT:", WORK_ROOT)
print("REPO_DIR:", REPO_DIR)

## 3 — Clone or update repository (public, no token)

In [ ]:
import subprocess

if REPO_DIR.is_dir() and (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=True)
    print("Updated existing clone.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    print("Cloned repository.")

os.chdir(REPO_DIR)
!git log -1 --oneline

## 4 — Install dependencies

In [ ]:
import os
os.chdir(REPO_DIR)
!pip install -q -r requirements.txt

## 5 — Colab environment (set before running experiments)

In [ ]:
import os

os.environ["THESIS_RUN_ENV"] = "colab"
os.environ["WANDB_DISABLED"] = "true"
os.environ.setdefault("THESIS_DELETE_MODELS_AFTER_TRAIN", "1")

print("THESIS_RUN_ENV =", os.environ["THESIS_RUN_ENV"])

## 6 — Check labeled dataset under `data/`

In [ ]:
from pathlib import Path

data_dir = REPO_DIR / "data"
for name in ("ner_dataset.pkl", "ner_dataset.csv", "ner_dataset.xlsx"):
    p = data_dir / name
    print(f"{name}: {'OK' if p.is_file() else 'MISSING'}")

if not any((data_dir / n).is_file() for n in ("ner_dataset.pkl", "ner_dataset.csv", "ner_dataset.xlsx")):
    raise FileNotFoundError(
        "Upload ner_dataset.pkl (preferred) to " + str(data_dir)
    )

## 7 — Run configuration

In [ ]:
from pathlib import Path

# Isolated run root — all artifacts, splits, seeds, and checkpoints live here only.
OUTPUT_DIR = WORK_ROOT / "5folds_run"
NUM_SEEDS = 10
MODELS = "dictabert,berel"
NER_DATASET_PKL = REPO_DIR / "data" / "ner_dataset_150_seed42.pkl"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set True once if training_seeds.json has fewer than NUM_SEEDS (e.g. after a 3-seed journal run).
REGENERATE_TRAINING_SEEDS = False
# Set True only to skip the 9-run Exp04 λ grid (uses defaults 10/5 or cached selection).
SKIP_EXP04_LOSS_CALIBRATION = False

print("OUTPUT_DIR:", OUTPUT_DIR)
print("NUM_SEEDS:", NUM_SEEDS)
print("NER_DATASET_PKL:", NER_DATASET_PKL)

## 7b — Preflight (GPU, corpus, seed file)

If cell 8 fails with `CalledProcessError`, **scroll up** in the output for the real `Traceback` / `[PREP]` line (the exception is only the exit code).

In [ ]:
import json
import torch
from pathlib import Path

if not NER_DATASET_PKL.is_file():
    raise FileNotFoundError(f"Missing corpus: {NER_DATASET_PKL} — run cell 3 (git pull) again.")

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARN: CPU runtime works but is very slow; use Runtime → GPU.")

seeds_path = Path(OUTPUT_DIR) / "training_seeds.json"
if seeds_path.is_file():
    stored = json.loads(seeds_path.read_text(encoding="utf-8")).get("seeds", [])
    print(f"training_seeds.json: {len(stored)} seeds on disk, NUM_SEEDS={NUM_SEEDS}")
    if len(stored) < NUM_SEEDS and not REGENERATE_TRAINING_SEEDS:
        raise ValueError(
            f"{seeds_path} has {len(stored)} seeds but NUM_SEEDS={NUM_SEEDS}. "
            "Set REGENERATE_TRAINING_SEEDS = True in cell 7 and re-run 7 → 7b → 8, "
            "or delete training_seeds.json under 5folds_run."
        )
else:
    print("training_seeds.json: will be created on first run.")

## 7c — Debug run (captures error text; dictabert only, optional skip λ grid)

Run this **once** if cell 8 fails with no visible error. Then fix and use cell 8 for the full run.

In [ ]:
import os, sys, subprocess

env = os.environ.copy()
env.update({
    "THESIS_RUN_ENV": "colab",
    "WANDB_DISABLED": "true",
    "THESIS_NER_CSV": str(NER_DATASET_PKL),
    "PYTHONUNBUFFERED": "1",
})
dbg_argv = [
    sys.executable, "run_cross_data_model_comparison.py",
    "--journal-paper", "--num-seeds", str(NUM_SEEDS),
    "--regenerate-training-seeds",
    "--resume", "--output-dir", str(OUTPUT_DIR),
    "--exp07-source", "rerun",
    "--models", "dictabert",
    "--base-mode", "auto",
    "--skip-exp04-loss-calibration",
]
proc = subprocess.run(dbg_argv, cwd=str(REPO_DIR), env=env, text=True, capture_output=True)
print("--- stdout ---\n", proc.stdout or "(empty)")
print("--- stderr ---\n", proc.stderr or "(empty)")
print("exit code:", proc.returncode)

## 8 — Run cross-comparison (argv one row per argument)

Uses **`--journal-paper`** with **`--num-seeds 10`**. Same command split into a Python list so each flag is its own line.

Expect a long run; use **`--resume`** on reconnect (re-run cells 1–7b, then this cell only).

In [ ]:
import os
import sys
import subprocess

env = os.environ.copy()
env["THESIS_RUN_ENV"] = "colab"
env["WANDB_DISABLED"] = "true"
env["THESIS_NER_CSV"] = str(NER_DATASET_PKL)

argv = [
    sys.executable,
    "run_cross_data_model_comparison.py",
    "--journal-paper",
    "--num-seeds",
    str(NUM_SEEDS),
    "--resume",
    "--output-dir",
    str(OUTPUT_DIR),
    "--exp07-source",
    "rerun",
    "--models",
    MODELS,
    "--base-mode",
    "auto",
]
if REGENERATE_TRAINING_SEEDS:
    argv.append("--regenerate-training-seeds")
if SKIP_EXP04_LOSS_CALIBRATION:
    argv.append("--skip-exp04-loss-calibration")

print("THESIS_NER_CSV =", env["THESIS_NER_CSV"])
print("Command:")
for i, part in enumerate(argv):
    print(f"  [{i}] {part}")

env["PYTHONUNBUFFERED"] = "1"
proc = subprocess.run(argv, cwd=str(REPO_DIR), env=env)
if proc.returncode != 0:
    raise RuntimeError(
        f"Runner exited with code {proc.returncode}. "
        "Read [FATAL] / Traceback lines above, or run cell 7c (debug capture)."
    )

## 9 — Optional: rebuild Excel/JSON only (no retraining)

In [ ]:
import os
import sys
import subprocess

os.chdir(REPO_DIR)

argv_rebuild = [
    sys.executable,
    "run_cross_data_model_comparison.py",
    "--rebuild-from-checkpoint",
    "--output-dir",
    str(OUTPUT_DIR),
]

subprocess.run(argv_rebuild, check=True)

## 10 — List key output files

In [ ]:
from pathlib import Path

out = Path(OUTPUT_DIR)
checks = [
    "run_manifest.json",
    "training_seeds.json",
    "data/ner_dataset_150_seed42.pkl",
    "exp07/splits/split_meta.json",
    "cross_comparison_progress_latest.json",
    "cross_comparison_latest.xlsx",
]
for rel in checks:
    p = out / rel
    print(f"{rel}: {'OK' if p.is_file() else 'missing'}")

for p in sorted(out.glob("consolidated_error_analysis*.xlsx"))[-2:]:
    print("workbook:", p.name)